# Priorização de inspeções de vegetação na Via Dutra

**Turma:** 2CCPG

| Integrante | RM |
|---|---:|
| Pedro Henrique dos Santos Cardoso | 563268 |
| Gabriel Gibin Leoncio | 565462 |
| Rafael do Nascimento Silva | 566263 |
| Rai Augusto Ribeiro | 562870 |
| Guilherme Morais de Assis | 564198 |
| Lucas Werpp Franco | 556044 |

Prova de conceito de Ciência de Dados com dados reais Sentinel-2. O notebook reproduz a análise de 92 pontos da BR-116 em 2024 e 2025.

**Pergunta:** quais pontos apresentam maior presença ou aumento de vegetação no entorno imediato e devem ser priorizados para inspeção?

As classes são indicadores de triagem, não laudos de risco.

## 1. Abrir o projeto no Colab ou no VS Code

No Colab, execute a próxima célula: o projeto será baixado automaticamente do GitHub, sem ZIP e sem Google Drive. No VS Code, basta abrir a pasta completa do projeto.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess

REPO_URL = 'https://github.com/gabrwell/monitoramento-vegetacao-via-dutra.git'
DESTINO_COLAB = Path('/content/monitoramento-vegetacao-via-dutra')

EM_COLAB = False
try:
    import google.colab
    EM_COLAB = True
except ImportError:
    print('Execução fora do Colab.')

def pasta_do_projeto(caminho):
    return caminho.is_dir() and (caminho / 'config/projeto_satelite.json').exists()

candidatas = [
    DESTINO_COLAB,
    Path.cwd(),
    *Path.cwd().parents,
]
PROJECT_DIR = next((p for p in candidatas if pasta_do_projeto(p)), None)
if PROJECT_DIR is None and EM_COLAB:
    print('Baixando o projeto do GitHub...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(DESTINO_COLAB)], check=True)
    PROJECT_DIR = DESTINO_COLAB
if PROJECT_DIR is None:
    raise FileNotFoundError('Abra a pasta completa do projeto no VS Code.')
if not pasta_do_projeto(PROJECT_DIR):
    raise FileNotFoundError('O repositório não contém a estrutura esperada do projeto.')
os.chdir(PROJECT_DIR)
print('Projeto:', PROJECT_DIR)

## 2. Dependências e configuração

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)
config = json.loads(Path('config/projeto_satelite.json').read_text(encoding='utf-8'))
display(config)

## 3. Coleta reproduzível

O projeto já inclui os CSVs reais usados no relatório. Altere `REFAZER_COLETA` para `True` somente se quiser consultar novamente a API pública do Planetary Computer.

In [ ]:
REFAZER_COLETA = False
if REFAZER_COLETA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, 'scripts/coletar_sentinel.py'], check=True)
else:
    print('Usando o dataset real já incluído no projeto.')

## 4. Auditoria do dataset real

In [ ]:
import pandas as pd
observacoes = pd.read_csv('data/processed/observacoes_sentinel.csv')
print('Linhas:', len(observacoes))
print('Pontos:', observacoes['ponto_id'].nunique())
print('Períodos:', sorted(observacoes['periodo'].astype(str).unique()))
display(observacoes.head())
display(observacoes.groupby(['periodo', 'data_aquisicao', 'tile']).agg(
    pontos=('ponto_id', 'count'), nuvens_pct=('nuvens_item_pct', 'first'),
    validade_media=('fracao_valida', 'mean')
))

## 5. Engenharia de atributos e rótulos

A análise calcula a variação temporal, aplica a regra fixa de prioridade e recria gráficos e resumo.

In [ ]:
subprocess.run([sys.executable, 'scripts/analisar_sentinel.py'], check=True)
segmentos = pd.read_csv('data/processed/segmentos_priorizados.csv')
resumo = json.loads(Path('outputs/resumo_resultados.json').read_text(encoding='utf-8'))
display(resumo)

## 6. Distribuição das prioridades

In [ ]:
from IPython.display import Image, display
display(Image(filename='outputs/figures/distribuicao_prioridade.png'))

## 7. Comparação temporal

In [ ]:
display(Image(filename='outputs/figures/comparacao_ndvi.png'))
display(Image(filename='outputs/figures/variacao_ndvi.png'))

## 8. Pontos que devem ser verificados primeiro

In [ ]:
colunas = ['ponto_id', 'longitude', 'latitude', 'prioridade_inspecao',
           'indice_prioridade_0_100', 'ndvi_p90_atual', 'delta_ndvi_media', 'criterio_rotulo']
display(segmentos[colunas].head(10))

## 9. Interpretação e limitações

Foram obtidas **184 observações reais**, correspondentes a **92 pontos em dois anos**. A regra encontrou **65 prioridades baixas, 22 médias e 5 altas**. O NDVI médio geral variou de 0,1504 para 0,1440; portanto, não houve aumento generalizado.

A resolução de 10 m não permite observar galhos, placas, árvores inclinadas ou invasão do acostamento. A janela mistura pista e terrenos vizinhos, e duas datas não definem uma tendência. Os cinco pontos altos são candidatos a inspeção, não riscos confirmados. A evolução exige validação de campo, histórico de manutenção e uma série temporal maior.

## 10. Conclusão

A prova de conceito demonstra um pipeline automatizado de aquisição, preparação, engenharia de atributos, rotulagem, análise e visualização usando dados públicos reais. O resultado é uma fila inicial de inspeção, acompanhada de critérios e limitações explícitos.